# Time series (PCMCI, PCMCI+, LPCMCI, Granger, VARMA-LiNGAM, longitudinal LiNGAM)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/time_series.ipynb)

Six methods find causes in data measured over time. PCMCI and Granger find the links from earlier
steps; PCMCI+ also finds links within one step, and LPCMCI allows series that are not measured.
VARMA-LiNGAM estimates the weights of the links within one step and from the step before.
Longitudinal LiNGAM takes a panel instead: the same units measured at a few time points. Each
returns a `TemporalStructure`, one graph per lag.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.5.0` from PyPI;
local kernels use the installed environment.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[numba,viz]==0.5.0" pandas

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

import andrey
import andrey.viz
from andrey.data import sample_scm

## Data

`sample_scm` draws one series of 1,000 time steps over 4 variables: a vector autoregression with
one lag, links within each step from a random DAG, and uniform noise. The drawing merges the lags
into one graph; a loop is a variable's dependence on its own past.

In [ ]:
series = sample_scm(
    graph="erdos_renyi",
    functional="var",
    noise="uniform",
    d=4,
    n=1000,
    seed=1,
    density=1.0,
)
X, truth = series.data, series.graph
print(f"{X.shape[0]} time steps of {X.shape[1]} variables")
print(truth)
display(andrey.viz.draw(truth.summary_graph()))

## Links from the past: PCMCI and Granger

PCMCI tests each lagged link with partial correlations, given the strongest earlier causes it
finds first; `tau_max` is the largest lag tested. Granger regresses each variable on the past
`maxlag` steps of every variable with a cross-validated lasso. Neither tests links within one step.

In [ ]:
print(andrey.pcmci(X, tau_max=1), end="\n\n")
print(andrey.granger(X, maxlag=1))

## Links within a step: PCMCI+ and LPCMCI

PCMCI+ adds the same-step links and orients them where the data allow; lag 0 is a CPDAG. LPCMCI
allows hidden series, so it returns a partial ancestral graph (PAG): a circle marks an end the data
leave open.

In [ ]:
print(andrey.pcmciplus(X, tau_max=1), end="\n\n")
print(andrey.lpcmci(X, tau_max=1))

## Weights: VARMA-LiNGAM

VARMA-LiNGAM uses the non-Gaussian noise to order the variables within a step, and estimates the
weight of every link. `prune=True` sets small weights to zero.

In [ ]:
out = andrey.varma_lingam(X, order=(1, 1), prune=True)
print(out, end="\n\n")
print("lag 0 weights, row = cause:")
print(np.round(out.structure.lag_weights[0], 2))

## Compare the links

Each method's links at lags 0 and 1 beside the true ones, a variable's dependence on its own past
included. A same-step link counts once, whatever its direction; a lagged link counts with its
direction. LPCMCI is left out: its lagged graphs keep only the links it orients as causes, so its
PAG above is the answer to read.

In [ ]:
def links(structure):
    found = set()
    for lag in (0, 1):
        if lag not in structure.lags:
            continue
        for i, j, _ in structure.lag(lag).oriented_edges():
            found.add((min(i, j), max(i, j), 0) if lag == 0 else (i, j, 1))
    return found


true_links = links(truth)
fits = {
    "PCMCI": andrey.pcmci(X, tau_max=1),
    "Granger": andrey.granger(X, maxlag=1),
    "PCMCI+": andrey.pcmciplus(X, tau_max=1),
    "VARMA-LiNGAM": andrey.varma_lingam(X, order=(1, 1), prune=True),
}
rows = {}
for name, fit in fits.items():
    found = links(fit.structure)
    rows[name] = {
        "true links found": f"{len(found & true_links)} of {len(true_links)}",
        "extra links": len(found - true_links),
    }
pd.DataFrame(rows).T

## Panels: longitudinal LiNGAM

Longitudinal LiNGAM takes one table per time point, the same units in the same rows. Here 500 units
are measured at 3 time points; within each, `x0 -> x1 -> x2`, and each variable depends on its own
value at the time point before.

In [ ]:
rng = np.random.default_rng(0)
b0 = np.array([[0.0, 0.0, 0.0], [0.8, 0.0, 0.0], [0.0, -0.7, 0.0]])  # row = effect
mix = np.linalg.inv(np.eye(3) - b0)
panel = [rng.uniform(-1, 1, (500, 3)) @ mix.T]
for _ in range(2):
    panel.append((0.5 * panel[-1] + rng.uniform(-1, 1, (500, 3))) @ mix.T)

out = andrey.longitudinal_lingam(panel, n_lags=1)
print(out, end="\n\n")
print("lag 0 at the last time point:", out.structure.lag(0).oriented_edges())

The [benchmarks](https://andrey.abel.ai/docs/benchmarks.html) compare each method with tigramite,
lingam, or causal-learn on the simulations of the methods' papers.

## Use your own data

Pass a table with one row per time step, such as a DataFrame whose column names become the graph's
labels, or change `seed` and `n` above and rerun.